# Backpressure & load shedding — the queueing arithmetic

Companion notebook for System Design, Plainly — Season 2, Part 5. Every number in the post is computed here.

In [ ]:
# The utilization cliff: M/M/1 time in system = S / (1 - rho).
# p99 is ~4.6x the mean for an exponential (ln 100).
import math
S = 10.0  # ms service time
for rho in (0.5, 0.8, 0.9, 0.95, 0.99):
    mean = S / (1 - rho)
    print(f"rho={rho:.2f}: mean {mean:7.1f} ms   p99 ~{4.6*mean:8.1f} ms")
assert abs(S/(1-0.9) - 100) < 1e-9
assert abs(S/(1-0.99) - 1000) < 1e-9
print("OK: 90% busy -> 10x the service time; 99% busy -> 100x")

In [ ]:
# Little's law: L = lambda * W. A dependency that gets 20x slower needs 20x the
# threads, connections and memory at the same arrival rate — nothing has to crash.
lam = 2000  # requests/second
for w_ms in (50, 100, 250, 500, 1000):
    print(f"W = {w_ms:4d} ms -> {lam * w_ms / 1000:,.0f} requests in flight")
assert lam * 0.05 == 100 and lam * 1.0 == 2000
# Sizing a queue from a latency budget: queue cap = drain rate * max wait.
mu, max_wait = 1000, 0.2  # 1,000 req/s, at most 200 ms of queueing
print(f"Queue cap from a {max_wait*1000:.0f} ms budget at {mu}/s: {int(mu*max_wait)} requests")
assert int(mu * max_wait) == 200

In [ ]:
# Bounded vs unbounded under a 2x overload (2,000 offered, 1,000 capacity, 60 s).
# Goodput = completions inside the 2 s client deadline.
def simulate(cap, bound, offered=2000, seconds=60, deadline=2.0, dt=0.05):
    q, served, shed, late = 0.0, 0, 0, 0
    for _ in range(int(seconds / dt)):
        arrivals = offered * dt
        wait = q / cap
        served_now = min(q + arrivals, cap * dt)
        if bound is not None:
            accepted = min(arrivals, max(0.0, bound - q) + cap * dt)
            shed += arrivals - accepted
            q = q + accepted - served_now
        else:
            q = q + arrivals - served_now
        if wait > deadline:
            late += served_now
        else:
            served += served_now
        q = max(q, 0.0)
    return served, shed, late

unb = simulate(1000, None)
bnd = simulate(1000, 200)
print(f"Unbounded: served in time {unb[0]:,.0f}, shed {unb[1]:,.0f}, served too late {unb[2]:,.0f}")
print(f"Bounded@200: served in time {bnd[0]:,.0f}, shed {bnd[1]:,.0f}, served too late {bnd[2]:,.0f}")
assert bnd[0] > unb[0] and bnd[2] < unb[2]
print("OK: the bounded queue serves MORE goodput — shedding is not lost work, it is triage")

In [ ]:
# The retry storm: blind retries multiply offered load.
# If a fraction f of requests is shed and every shed request is retried up to N times,
# effective load = 1 + f + f^2 + ... + f^N (each retry faces the same shed fraction).
def retry_multiplier(shed_frac, max_retries):
    return sum(shed_frac ** k for k in range(max_retries + 1))

for f in (0.1, 0.3, 0.5):
    for n in (1, 3):
        print(f"shed {f:.0%}, {n} retr{'y' if n==1 else 'ies'}: offered load x{retry_multiplier(f, n):.2f}")
assert abs(retry_multiplier(0.5, 3) - 1.875) < 1e-9
# The fix: exponential backoff + jitter, and a Retry-After header so clients wait.
# With backoff, retries spread over seconds instead of landing in the same instant.
import random
rng = random.Random(7)
delays = sorted(min(2.0, 0.1 * 2**k) * rng.uniform(0.5, 1.5) for k in range(5))
print("Backoff delays (s) for 5 retries:", [round(d, 2) for d in delays])

## Takeaways

1. The utilization cliff is brutal and exact: at 90% busy you wait 10x the service time; at 99%, 100x. Keep steady-state utilization below ~80% or the queue owns you.
2. Little's law turns a latency problem into a capacity problem: a 20x slower dependency is 20x the in-flight requests at the same arrival rate.
3. A queue only buys time, never capacity. Size it from the latency budget (cap = drain rate × max wait), and shed the rest fast with a 429/503 + Retry-After.
4. Blind retries are a second overload: shedding half with 3 blind retries turns 2,000 req/s into ~3,750. Backoff and jitter are part of the shedding design, not an afterthought.